# GDBA-Net Testing
Run setup cells in order. Edit the final cell for each experiment.

In [1]:
!nvidia-smi

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Mon Oct 12 23:19:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content

from pathlib import Path
import subprocess

REPO_DIR = Path("/content/GDBA_Net")
REPO_URL = "https://github.com/biplabpoudel25/GDBA_Net.git"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Using existing repository.")

%cd /content/GDBA_Net

/content
Using existing repository.
/content/GDBA_Net


In [4]:
%pip install \
    hydra-core \
    tensorboardX==2.6.2.2 \
    einops \
    numpy==1.26.4 \
    Pillow \
    PyYAML==6.0.2 \
    tqdm==4.67.1 \
    opencv-python==4.11.0.86 \
    PyWavelets \
    scipy \
    scikit-learn \
    matplotlib

In [5]:
from pathlib import Path

repo = Path("/content/GDBA_Net")
for filename in ["test.py", "models/sam.py", "configs/gdba_net.yaml"]:
    path = repo / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing repository file: {filename}")
    print("Found:", filename)

Found: test.py
Found: models/sam.py
Found: configs/gdba_net.yaml


In [6]:
from pathlib import Path

for filename in ["models/mmseg/__init__.py", "models/mmseg/models/__init__.py"]:
    path = Path("/content/GDBA_Net") / filename
    if path.exists():
        path.write_text("# Colab compatibility: disable unused legacy MMseg imports\\n")
        print("Compatibility fix applied:", filename)

Compatibility fix applied: models/mmseg/__init__.py
Compatibility fix applied: models/mmseg/models/__init__.py


In [7]:
from pathlib import Path
import yaml

config_path = Path("/content/GDBA_Net/configs/gdba_net.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config["test_dataset"]["batch_size"] = 1
config["test_dataset"]["num_workers"] = 2

with open(config_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Test batch size:", config["test_dataset"]["batch_size"])
print("Test workers:", config["test_dataset"]["num_workers"])

Test batch size: 1
Test workers: 2


In [8]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/GDBA_Net_Colab/pretrained/sam2_hiera_large.pt"
)

destination = Path(
    "/content/gdba_data/pretrained/sam2_hiera_large.pt"
)

if not source.is_file():
    raise FileNotFoundError(f"Pretrained checkpoint not found: {source}")

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

print("Pretrained checkpoint ready.")
print(f"Size: {destination.stat().st_size / 1024**2:.1f} MB")

Pretrained checkpoint ready.
Size: 856.4 MB


## Test dataset
Place the checkpoint and dataset in accessible runtime folders. Change the paths below, then duplicate this cell for additional experiments.

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/CryoNuSeg_Combined/test/images" \
    --test_labels "/content/drive/MyDrive/CryoNuSeg_Combined/test/masks" \
    --save_masks \
    --output_dir /content/gdba_test_results/CryoNuSeg_Combined_Test

/content/GDBA_Net
Using device: cuda
Test dataset: 10 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.8481
Iou: 0.7381
Precision: 0.8519
Recall: 0.8476
Specificity: 0.9286

Results saved to: /content/gdba_test_results/CryoNuSeg_Combined_Test


# CryoNuSeg trained on Combined CPM as test dataset

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/Combined_CPM/images" \
    --test_labels "/content/drive/MyDrive/Combined_CPM/masks" \
    --save_masks \
    --output_dir "/content/gdba_test_results/CryoNuSeg_to_Combined_CPM"

/content/GDBA_Net
Using device: cuda
Test dataset: 79 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.8266
Iou: 0.7126
Precision: 0.8744
Recall: 0.8007
Specificity: 0.9706

Results saved to: /content/gdba_test_results/CryoNuSeg_to_Combined_CPM


# CryoNuSeg trained on TNBC as test dataset

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/TNBC_Processed/images" \
    --test_labels "/content/drive/MyDrive/TNBC_Processed/masks" \
    --save_masks \
    --output_dir "/content/gdba_test_results/CryoNuSeg_to_TNBC"

/content/GDBA_Net
Using device: cuda
Test dataset: 50 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.8045
Iou: 0.6829
Precision: 0.8271
Recall: 0.7952
Specificity: 0.9723

Results saved to: /content/gdba_test_results/CryoNuSeg_to_TNBC


# CryoNuSeg trained on CoNSeP as test dataset

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/CoNSeP_Combined/images" \
    --test_labels "/content/drive/MyDrive/CoNSeP_Combined/masks" \
    --save_masks \
    --output_dir "/content/gdba_test_results/CryoNuSeg_to_CoNSeP"

/content/GDBA_Net
Using device: cuda
Test dataset: 41 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.6557
Iou: 0.5051
Precision: 0.6649
Recall: 0.7134
Specificity: 0.9368

Results saved to: /content/gdba_test_results/CryoNuSeg_to_CoNSeP


# MoNuSeg trained and tested on MoNuSeg

In [17]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/test/images \
    --test_labels /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/test/masks \
    --save_masks \
    --output_dir /content/gdba_test_results/monuseg_train_test

/content/GDBA_Net
Using device: cuda
Test dataset: 11 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.8408
Iou: 0.7259
Precision: 0.8388
Recall: 0.8481
Specificity: 0.9516

Results saved to: /content/gdba_test_results/monuseg_train_test


# MoNuSeg trained and tested on TNBC

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/TNBC_Processed/images" \
    --test_labels "/content/drive/MyDrive/TNBC_Processed/masks" \
    --save_masks \
    --output_dir "/content/gdba_test_results/MoNuSeg_to_TNBC_Processed"

/content/GDBA_Net
Using device: cuda
Test dataset: 50 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.8146
Iou: 0.6936
Precision: 0.8127
Recall: 0.8359
Specificity: 0.9715

Results saved to: /content/gdba_test_results/MoNuSeg_to_TNBC_Processed


# MoNuSeg trained and tested on CoNSeP

In [ ]:
%cd /content/GDBA_Net

!python test.py \
    --config configs/gdba_net.yaml \
    --model /content/drive/MyDrive/GDBA_Net_Colab/checkpoints/model_epoch_best.pth \
    --test_images "/content/drive/MyDrive/CoNSeP_Combined/images" \
    --test_labels "/content/drive/MyDrive/CoNSeP_Combined/masks" \
    --save_masks \
    --output_dir "/content/gdba_test_results/MoNuSeg_to_CoNSeP_Combined"

/content/GDBA_Net
Using device: cuda
Test dataset: 41 samples
Inside handcrafted features
                                                                   
Average Test Metrics:
Dice: 0.6833
Iou: 0.5351
Precision: 0.7253
Recall: 0.6959
Specificity: 0.9565

Results saved to: /content/gdba_test_results/MoNuSeg_to_CoNSeP_Combined
